In [1]:
def main(datasources, start_date, end_date):
    """
    BigAlpha2026 \u4f20\u7edf\u91cf\u5316\u8d5b\u9053\u5355\u56e0\u5b50\uff1aDODI\uff08\u6df1\u5ea6\u6863\u4f4d\u89c4\u6a21\u504f\u79bb\u6307\u6570 Deep Order Disparity Index\uff09\u3002

    \u901a\u8fc7\u6bd4\u8f83\u76d8\u53e3\u8868\u9762(L1)\u4e0e\u6df1\u5c42(L3-L5)\u6bcf\u7b14\u59d4\u6258\u5e73\u5747\u89c4\u6a21\u7684\u5dee\u5f02\uff0c\u8bc6\u522b\u673a\u6784\u4ea4\u6613\u8005
    \u4ee5\u9650\u4ef7\u5355\u201c\u51b0\u5c71\u201d\u6a21\u5f0f\u9690\u853d\u5efa\u4ed3/\u51fa\u8d27\u7684\u75d5\u8ff9\u3002
    \u8fd4\u56de\u503c\uff1adate(datetime)\u3001instrument(string)\u3001factor(float)\u4e09\u5217\u3002
    """
    import numpy as np
    import pandas as pd
    import dai

    if not hasattr(dai, "query"):
        return pd.DataFrame(columns=["date", "instrument", "factor"])

    bar1m = datasources["bar1m"]

    # ------------------------------------------------------------------
    # DAI SQL CTE \u94fe\uff1a\u5728\u6570\u636e\u5e93\u5c42\u5b8c\u6210\u5206\u949f\u7ea7DODI\u8ba1\u7b97\uff0c
    # \u4ec5\u8fd4\u56de\u65e5\u9891\u805a\u5408\u7ed3\u679c\uff0c\u9ad8\u6548\u8282\u7701\u5185\u5b58\u3002
    # ------------------------------------------------------------------

    sql = f"""
    WITH level_sizes AS (
        SELECT
            date,
            instrument,
            strftime(date, '%Y-%m-%d') AS trading_day,
            -- \u6bcf\u6863\u5e73\u5747\u59d4\u91cf = \u59d4\u91cf / MAX(\u59d4\u6258\u7b14\u6570, 1) \u907f\u514d\u9664\u96f6
            CAST(bid_volume1 AS DOUBLE) / GREATEST(CAST(bid_num_orders1 AS DOUBLE), 1.0) AS bid_avg1,
            CAST(bid_volume2 AS DOUBLE) / GREATEST(CAST(bid_num_orders2 AS DOUBLE), 1.0) AS bid_avg2,
            CAST(bid_volume3 AS DOUBLE) / GREATEST(CAST(bid_num_orders3 AS DOUBLE), 1.0) AS bid_avg3,
            CAST(bid_volume4 AS DOUBLE) / GREATEST(CAST(bid_num_orders4 AS DOUBLE), 1.0) AS bid_avg4,
            CAST(bid_volume5 AS DOUBLE) / GREATEST(CAST(bid_num_orders5 AS DOUBLE), 1.0) AS bid_avg5,
            CAST(ask_volume1 AS DOUBLE) / GREATEST(CAST(ask_num_orders1 AS DOUBLE), 1.0) AS ask_avg1,
            CAST(ask_volume2 AS DOUBLE) / GREATEST(CAST(ask_num_orders2 AS DOUBLE), 1.0) AS ask_avg2,
            CAST(ask_volume3 AS DOUBLE) / GREATEST(CAST(ask_num_orders3 AS DOUBLE), 1.0) AS ask_avg3,
            CAST(ask_volume4 AS DOUBLE) / GREATEST(CAST(ask_num_orders4 AS DOUBLE), 1.0) AS ask_avg4,
            CAST(ask_volume5 AS DOUBLE) / GREATEST(CAST(ask_num_orders5 AS DOUBLE), 1.0) AS ask_avg5,
            -- \u5404\u6863\u59d4\u91cf\u7528\u4e8e\u91cf\u52a0\u6743\u6c42\u548c
            CAST(bid_volume1 AS DOUBLE) AS bv1, CAST(bid_volume2 AS DOUBLE) AS bv2,
            CAST(bid_volume3 AS DOUBLE) AS bv3, CAST(bid_volume4 AS DOUBLE) AS bv4,
            CAST(bid_volume5 AS DOUBLE) AS bv5,
            CAST(ask_volume1 AS DOUBLE) AS av1, CAST(ask_volume2 AS DOUBLE) AS av2,
            CAST(ask_volume3 AS DOUBLE) AS av3, CAST(ask_volume4 AS DOUBLE) AS av4,
            CAST(ask_volume5 AS DOUBLE) AS av5
        FROM {bar1m}
        WHERE bid_price1 > 0 AND ask_price1 > 0
    ),
    deep_composite AS (
        SELECT
            *,
            -- \u3010\u76d8\u53e3\uff0d\u6df1\u5c42\u59d4\u91cf\u52a0\u6743\u5e73\u5747\u89c4\u6a21\u3011
            -- \u673a\u6784\u9650\u4ef7\u5355\u5f80\u5f80\u4f7f\u7528\u5b8c\u6574\u80a1\u4efd\u6570\uff0c\u50ac\u751f\u5927\u5355\u7b14\u59d4\u91cf\uff1b
            -- \u6563\u6237\u62a5\u4ef7\u96f6\u6563\u3001\u5355\u7b14\u91cf\u5c0f\u3002\u8fd9\u4e2a\u5dee\u5f02\u5728\u6df1\u5c42(L3-L5)\u66f4\u660e\u663e\u3002
            -- \u91cf\u52a0\u6743\u907f\u514d\u6781\u7aef\u7a7a\u6863\u4f4d\u5e72\u6270\uff0c\u53ea\u6709\u59d4\u91cf\u8f83\u5927\u7684\u6863\u4f4d\u624d\u663e\u8457\u8d21\u732e\u3002
            CASE WHEN (bv3 + bv4 + bv5) > 1e-8
                THEN (bv3 * bid_avg3 + bv4 * bid_avg4 + bv5 * bid_avg5)
                     / (bv3 + bv4 + bv5)
            END AS bid_deep_avg,
            CASE WHEN (av3 + av4 + av5) > 1e-8
                THEN (av3 * ask_avg3 + av4 * ask_avg4 + av5 * ask_avg5)
                     / (av3 + av4 + av5)
            END AS ask_deep_avg,
            -- \u5168\u91cf\u4e94\u6863\u59d4\u4e70\u59d4\u5356\u5404\u81ea\u603b\u91cf
            (bv1 + bv2 + bv3 + bv4 + bv5) AS total_bid,
            (av1 + av2 + av3 + av4 + av5) AS total_ask
        FROM level_sizes
    ),
    signals AS (
        SELECT
            *,
            -- \u3010\u76d8\u53e3\uff0d\u6df1\u5c42\u89c4\u6a21\u504f\u79bb\u3011
            -- \u8868\u9762(L1)\u4e0e\u6df1\u5c42(L3-L5)\u7684\u76f8\u5bf9\u504f\u5dee\uff08\u6709\u754c\u5f52\u4e00\u5316 [-1, 1]\uff09
            -- bid_div > 0 = \u6df1\u5c42\u4e70\u4fa7\u6bcf\u5355\u89c4\u6a21 > \u8868\u9762\u4e70\u4fa7 = \u673a\u6784\u79ef\u7d2f
            -- ask_div > 0 = \u6df1\u5c42\u5356\u4fa7\u6bcf\u5355\u89c4\u6a21 > \u8868\u9762\u5356\u4fa7 = \u673a\u6784\u6d3e\u53d1
            CASE WHEN bid_deep_avg IS NOT NULL AND bid_avg1 IS NOT NULL
                THEN (bid_deep_avg - bid_avg1)
                     / GREATEST(bid_deep_avg + bid_avg1, 1.0)
            END AS bid_div,
            CASE WHEN ask_deep_avg IS NOT NULL AND ask_avg1 IS NOT NULL
                THEN (ask_deep_avg - ask_avg1)
                     / GREATEST(ask_deep_avg + ask_avg1, 1.0)
            END AS ask_div,
            -- \u3010\u76d8\u53e3\uff0d\u59d4\u91cf\u4e0d\u5e73\u8861\u3011
            -- \u5168\u91cf\u4e94\u6863\u59d4\u4e70\u603b\u91cf - \u59d4\u5356\u603b\u91cf / \u603b\u91cf\uff08\u6709\u754c [-1, 1]\uff09
            CASE WHEN (total_bid + total_ask) > 1e-8
                THEN (total_bid - total_ask) / (total_bid + total_ask)
                ELSE 0.0
            END AS book_imb
        FROM deep_composite
        WHERE bid_avg1 > 0 AND ask_avg1 > 0
    ),
    minute_signals AS (
        SELECT
            *,
            -- \u3010\u76d8\u53e3\uff0dDODI\u5206\u949f\u4fe1\u53f7\u3011
            -- \u51c0\u504f\u79bb = \u4e70\u4fa7\u76f8\u5bf9\u89c4\u6a21 - \u5356\u4fa7\u76f8\u5bf9\u89c4\u6a21
            -- \u6b63\u503c = \u4e70\u4fa7\u6df1\u5c42\u89c4\u6a21\u4f18\u52bf > \u5356\u4fa7\u6df1\u5c42\u89c4\u6a21\u4f18\u52bf = \u5438\u7b79\u4fe1\u53f7
            -- \u4ee5book_imb\u7f6e\u4fe1\u5ea6\u8c03\u6574\uff1a\u59d4\u91cf\u5931\u8861\u8d8a\u5c0f(= \u76d8\u53e3\u6700\u5e73\u8861)\u65f6\u4fe1\u53f7\u8d8a\u7eaf
            (COALESCE(bid_div, 0.0) - COALESCE(ask_div, 0.0))
            * (1.0 - ABS(COALESCE(book_imb, 0.0))) AS dodi_minute,
            -- \u5185\u65f6\u65f6\u6bb5\u5212\u5206\uff1aA\u80a1\u4ea4\u6613 9:30-11:30, 13:00-15:00
            -- early: 9:30-10:30 (\u5f00\u76d8\u6ce2\u52a8\u671f)
            -- mid: 10:30-11:30 + 13:00-14:00 (\u4e3b\u4ea4\u6613\u671f)
            -- late: 14:00-15:00 (\u5c3e\u76d8\u671f)
            CASE
                WHEN date::TIME <= TIME '10:30:00' THEN 'early'
                WHEN date::TIME <= TIME '14:00:00' THEN 'mid'
                ELSE 'late'
            END AS session
        FROM signals
    ),
    daily AS (
        SELECT
            CAST(trading_day AS DATETIME) AS date,
            instrument,
            COUNT(*) AS valid_minutes,
            -- \u3010\u65f6\u5e8f\u805a\u5408\uff0d\u7edf\u8ba1\u5047\u8bbe\u3011
            -- \u53d6\u76d8\u4e2d\u6bb5(10:30-14:00)\u5747\u503c\u4f5c\u4e3a\u6838\u5fc3\u4fe1\u53f7\uff0c
            -- \u6b64\u65f6\u6bb5\u6210\u4ea4\u91cf\u76f8\u5bf9\u7a33\u5b9a\uff0c\u673a\u6784\u884c\u4e3a\u4e0d\u6613\u88ab\u5f00\u76d8\u566a\u97f3/\u5c3e\u76d8\u535a\u5f08\u6c61\u67d3\u3002
            -- \u5c3e\u76d8 - \u65e9\u76d8\u8d8b\u52bf\u4f5c\u4e3a\u65b9\u5411\u786e\u8ba4\uff1a\u8d8b\u52bf\u4e3a\u6b63\u8868\u793a\u673a\u6784\u4fe1\u53f7\u5728\u65e5\u5185\u589e\u5f3a\u3002
            AVG(CASE WHEN session = 'mid' THEN dodi_minute END) AS mid_mean,
            AVG(CASE WHEN session = 'early' THEN dodi_minute END) AS early_mean,
            AVG(CASE WHEN session = 'late' THEN dodi_minute END) AS late_mean,
            COUNT(CASE WHEN session = 'mid' THEN 1 END) AS mid_cnt
        FROM minute_signals
        GROUP BY trading_day, instrument
    )
    SELECT
        date,
        instrument,
        -- \u65e5\u5185\u805a\u5408\uff1a\u76d8\u4e2d\u6838\u5fc3\u4fe1\u53f7 + 0.3 * (\u5c3e\u76d8 - \u65e9\u76d8)\u8d8b\u52bf
        -- \u8986\u76d6\u7387\u7f29\u6743 n/(n+30)\uff1a\u65e5\u5185\u6709\u6548\u5206\u949f\u6570\u4e0d\u8db3\u65f6\u964d\u6743
        (COALESCE(mid_mean, 0.0)
         + 0.3 * (COALESCE(late_mean, 0.0) - COALESCE(early_mean, 0.0)))
        * CAST(COALESCE(mid_cnt, 0) AS DOUBLE)
        / (CAST(COALESCE(mid_cnt, 0) AS DOUBLE) + 30.0) AS raw_factor
    FROM daily
    WHERE valid_minutes >= 120
    ORDER BY date, instrument
    """

    raw = dai.query(sql, filters={"date": [start_date, end_date]}, compression=True).df()

    if raw.empty:
        return pd.DataFrame(columns=["date", "instrument", "factor"])

    # --------------------------------------------------------------
    # \u80a1\u7968\u6c60\u5408\u5e76\uff08\u7b5b\u9009\u4e3a\u4e2d\u8bc11000\u6210\u5206\u80a1\uff09
    # --------------------------------------------------------------
    pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()

    raw["date"] = pd.to_datetime(raw["date"])
    pool["date"] = pd.to_datetime(pool["date"])
    raw["instrument"] = raw["instrument"].astype("string")
    pool["instrument"] = pool["instrument"].astype("string")

    df = pd.merge(raw, pool, how="inner", on=["date", "instrument"])
    df["raw_factor"] = pd.to_numeric(df["raw_factor"], errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["raw_factor"])

    if df.empty:
        return pd.DataFrame(columns=["date", "instrument", "factor"])

    # --------------------------------------------------------------
    # \u3010\u622a\u9762\u5904\u7406\uff0d\u7a33\u5065\u6807\u51c6\u5316\u3011
    # MAD\u53bb\u6781\u503c (5\u500dMAD) \u9002\u5e94\u4e0d\u540c\u5e02\u573a\u72b6\u6001\uff0c
    # \u968f\u540eZ-score\u6d88\u9664\u622a\u9762\u91cf\u7eb2\u3002
    # \u4e0d\u5f15\u5165\u6536\u76ca/\u884c\u4e1a/\u672a\u6765\u4fe1\u606f\uff0c\u65e0\u524d\u89c6\u504f\u5dee\u3002
    # --------------------------------------------------------------

    def _mad_zscore(s):
        med = s.median()
        mad = (s - med).abs().median()
        if np.isfinite(mad) and mad > 1e-12:
            w = s.clip(med - 5.0 * 1.4826 * mad, med + 5.0 * 1.4826 * mad)
        else:
            w = s.copy()
        std = w.std(ddof=0)
        if not np.isfinite(std) or std < 1e-12:
            return pd.Series(0.0, index=s.index, dtype="float64")
        return (w - w.mean()) / std

    df["factor"] = df.groupby("date", group_keys=False)["raw_factor"].transform(_mad_zscore)

    # \u3010\u65b9\u5411\u6821\u51c6\u3011\u66f4\u9ad8\u503c = \u6df1\u5c42\u4e70\u4fa7\u76f8\u5bf9\u89c4\u6a21\u66f4\u5927 = \u673a\u6784\u9690\u6027\u5438\u7b79\u4fe1\u53f7\uff0c
    # \u6309\u201c\u51b0\u5c71\u8ba2\u5355\u672a\u6d88 \u2192 \u672a\u6765\u4e0a\u6da8\u63a8\u529b\u201d\u903b\u8f91\uff0c\u6b63\u5411\u5bf9\u9f50\u6536\u76ca\u3002

    out = df.loc[:, ["date", "instrument", "factor"]].copy()
    out["factor"] = out["factor"].astype("float64")
    out = out.replace([np.inf, -np.inf], np.nan).dropna(subset=["factor"])

    return out.sort_values(["date", "instrument"]).reset_index(drop=True)